
# TP2 — Smoking Prediction
## 02. Model Training & Evaluation

### Objetivo

Entrenar y comparar modelos de clasificación binaria para predecir la variable `smoking`, utilizando los datos preparados durante la etapa exploratoria.

La evaluación se centrará en el **F1-score de la clase positiva (`smoking = 1`)**, que combina precisión y recall.

### Estrategia metodológica

- Separar los datos etiquetados mediante una partición estratificada y reproducible.
- Establecer un modelo de referencia basado en variables demográficas.
- Compararlo con modelos que incorporen información clínica.
- Evaluar el rendimiento mediante métricas de clasificación y seleccionar el modelo final.
- Conservar el modelo y su preprocesamiento para aplicar el mismo procedimiento al conjunto sin etiquetar.

La selección del modelo se realizará exclusivamente con datos etiquetados, evitando utilizar el conjunto destinado a predicción para evaluar su rendimiento.


In [10]:

from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# Cargar los datos preparados en la Notebook 01
DATA_DIR = Path("../data/processed")
df = pd.read_csv(DATA_DIR / "train_clean.csv")

# Separar predictores y variable objetivo
X = df.drop(columns=["ID", "smoking"])
y = df["smoking"]

# Reservar 20% para evaluación final
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# Dividir el 80% restante en entrenamiento y validación
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    stratify=y_temp,
    random_state=42
)

print("Entrenamiento:", X_train.shape)
print("Validación:", X_val.shape)
print("Prueba:", X_test.shape)

print("\nProporción de fumadores:")
print(f"Entrenamiento: {y_train.mean():.2%}")
print(f"Validación: {y_val.mean():.2%}")
print(f"Prueba: {y_test.mean():.2%}")


Entrenamiento: (30000, 25)
Validación: (10000, 25)
Prueba: (10000, 25)

Proporción de fumadores:
Entrenamiento: 36.66%
Validación: 36.66%
Prueba: 36.66%


In [11]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Identificar variables según su tipo
columnas_categoricas = X_train.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

columnas_numericas = X_train.select_dtypes(
    include="number"
).columns.tolist()

# Preprocesamiento para modelos de clasificación
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), columnas_numericas),
        ("categorical", OneHotEncoder(
            handle_unknown="ignore"
        ), columnas_categoricas)
    ]
)

print("Variables numéricas:", len(columnas_numericas))
print("Variables categóricas:", columnas_categoricas)
print("Total de predictores:", X_train.shape[1])


Variables numéricas: 22
Variables categóricas: ['gender', 'oral', 'tartar']
Total de predictores: 25


In [12]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score

# Variables demográficas para el modelo de referencia
variables_demograficas = [
    "gender", "age", "height(cm)", "weight(kg)"
]

preprocessor_base = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(),
         ["age", "height(cm)", "weight(kg)"]),
        ("categorical", OneHotEncoder(handle_unknown="ignore"),
         ["gender"])
    ]
)

modelo_base = Pipeline([
    ("preprocessing", preprocessor_base),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Entrenar exclusivamente con el conjunto de entrenamiento
modelo_base.fit(X_train[variables_demograficas], y_train)

# Evaluar sobre validación
pred_base = modelo_base.predict(X_val[variables_demograficas])

print("MODELO BASE — REGRESIÓN LOGÍSTICA")
print(f"F1-score (clase 1): {f1_score(y_val, pred_base):.4f}")
print(f"Precision: {precision_score(y_val, pred_base):.4f}")
print(f"Recall: {recall_score(y_val, pred_base):.4f}")


MODELO BASE — REGRESIÓN LOGÍSTICA
F1-score (clase 1): 0.7028
Precision: 0.5552
Recall: 0.9574


In [13]:

# Regresión logística con variables demográficas y clínicas

modelo_completo = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

modelo_completo.fit(X_train, y_train)

pred_completo = modelo_completo.predict(X_val)

resultados = pd.DataFrame({
    "Modelo": [
        "Regresión logística - Demográfico",
        "Regresión logística - Completo"
    ],
    "F1-score": [
        f1_score(y_val, pred_base),
        f1_score(y_val, pred_completo)
    ],
    "Precision": [
        precision_score(y_val, pred_base),
        precision_score(y_val, pred_completo)
    ],
    "Recall": [
        recall_score(y_val, pred_base),
        recall_score(y_val, pred_completo)
    ]
})

display(resultados.round(4))


,Modelo,F1-score,Precision,Recall
0,Regresión logística - Demográfico,0.7028,0.5552,0.9574
1,Regresión logística - Completo,0.6578,0.6409,0.6757


In [14]:

import numpy as np

# Probabilidades de pertenecer a la clase positiva
probabilidades = modelo_completo.predict_proba(X_val)[:, 1]

umbrales = np.arange(0.20, 0.81, 0.01)

resultados_umbral = pd.DataFrame([
    {
        "Umbral": round(float(umbral), 2),
        "F1-score": f1_score(
            y_val,
            (probabilidades >= umbral).astype(int)
        )
    }
    for umbral in umbrales
])

mejor_fila = resultados_umbral.loc[
    resultados_umbral["F1-score"].idxmax()
]

mejor_umbral = float(mejor_fila["Umbral"])

print(f"Mejor umbral: {mejor_umbral:.2f}")
print(f"F1-score optimizado: {mejor_fila['F1-score']:.4f}")
print(f"F1-score modelo base: {f1_score(y_val, pred_base):.4f}")


Mejor umbral: 0.39
F1-score optimizado: 0.7076
F1-score modelo base: 0.7028


In [15]:

# Optimizar el umbral del modelo demográfico

prob_base = modelo_base.predict_proba(
    X_val[variables_demograficas]
)[:, 1]

resultados_base_umbral = pd.DataFrame([
    {
        "Umbral": round(float(u), 2),
        "F1-score": f1_score(
            y_val, (prob_base >= u).astype(int)
        )
    }
    for u in umbrales
])

mejor_base = resultados_base_umbral.loc[
    resultados_base_umbral["F1-score"].idxmax()
]

print("MODELOS CON UMBRALES OPTIMIZADOS")
print(
    f"Demográfico: F1={mejor_base['F1-score']:.4f} "
    f"| Umbral={mejor_base['Umbral']:.2f}"
)
print(
    f"Completo:    F1={mejor_fila['F1-score']:.4f} "
    f"| Umbral={mejor_umbral:.2f}"
)


MODELOS CON UMBRALES OPTIMIZADOS
Demográfico: F1=0.7030 | Umbral=0.48
Completo:    F1=0.7076 | Umbral=0.39


In [16]:

from sklearn.ensemble import RandomForestClassifier

modelo_rf = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    ))
])

modelo_rf.fit(X_train, y_train)

prob_rf = modelo_rf.predict_proba(X_val)[:, 1]

resultados_rf = pd.DataFrame([
    {
        "Umbral": round(float(u), 2),
        "F1-score": f1_score(
            y_val, (prob_rf >= u).astype(int)
        )
    }
    for u in umbrales
])

mejor_rf = resultados_rf.loc[
    resultados_rf["F1-score"].idxmax()
]

print("RANDOM FOREST")
print(f"Mejor umbral: {mejor_rf['Umbral']:.2f}")
print(f"F1-score: {mejor_rf['F1-score']:.4f}")
print(f"F1-score regresión logística: {mejor_fila['F1-score']:.4f}")


RANDOM FOREST
Mejor umbral: 0.41
F1-score: 0.7368
F1-score regresión logística: 0.7076


In [17]:

# Random Forest con variables demográficas

modelo_rf_base = Pipeline([
    ("preprocessing", preprocessor_base),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    ))
])

modelo_rf_base.fit(
    X_train[variables_demograficas], y_train
)

prob_rf_base = modelo_rf_base.predict_proba(
    X_val[variables_demograficas]
)[:, 1]

f1_rf_base = [
    f1_score(y_val, (prob_rf_base >= u).astype(int))
    for u in umbrales
]

indice = np.argmax(f1_rf_base)
umbral_rf_base = round(float(umbrales[indice]), 2)

print("RANDOM FOREST — COMPARACIÓN")
print(f"Demográfico: F1={f1_rf_base[indice]:.4f} | Umbral={umbral_rf_base}")
print(f"Completo:    F1={mejor_rf['F1-score']:.4f} | Umbral={mejor_rf['Umbral']:.2f}")


RANDOM FOREST — COMPARACIÓN
Demográfico: F1=0.7033 | Umbral=0.28
Completo:    F1=0.7368 | Umbral=0.41


In [18]:

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

# Evaluación final del modelo seleccionado
umbral_final = float(mejor_rf["Umbral"])

prob_test = modelo_rf.predict_proba(X_test)[:, 1]
pred_test = (prob_test >= umbral_final).astype(int)

print("EVALUACIÓN FINAL — RANDOM FOREST")
print(f"Umbral seleccionado: {umbral_final:.2f}")
print(f"F1-score clase 1: {f1_score(y_test, pred_test):.4f}")

print("\nReporte de clasificación:")
print(classification_report(y_test, pred_test, digits=4))

print("Matriz de confusión:")
print(confusion_matrix(y_test, pred_test))


EVALUACIÓN FINAL — RANDOM FOREST
Umbral seleccionado: 0.41
F1-score clase 1: 0.7415

Reporte de clasificación:
              precision    recall  f1-score   support

           0     0.9114    0.7144    0.8010      6334
           1     0.6407    0.8800    0.7415      3666

    accuracy                         0.7751     10000
   macro avg     0.7760    0.7972    0.7712     10000
weighted avg     0.8122    0.7751    0.7792     10000

Matriz de confusión:
[[4525 1809]
 [ 440 3226]]



### Resultados y conclusiones

Se compararon modelos de regresión logística y Random Forest utilizando una partición estratificada de entrenamiento (60 %), validación (20 %) y prueba (20 %).

El mejor modelo fue **Random Forest con todas las variables predictoras**, utilizando un umbral de clasificación de 0.41, seleccionado exclusivamente mediante el conjunto de validación.

En la evaluación final, el modelo obtuvo:

- **F1-score (clase 1):** 0.7415
- **Precision:** 0.6407
- **Recall:** 0.8800
- **Accuracy:** 0.7751

La hipótesis exploratoria recibió apoyo en validación: al incorporar las variables clínicas, el F1-score de Random Forest aumentó de 0.7033 a 0.7368, manteniendo el mismo algoritmo y optimizando los umbrales de ambos modelos.

El resultado final evidencia una elevada capacidad para identificar fumadores, aunque persisten falsos positivos. Por ello, el modelo debe interpretarse como una herramienta predictiva experimental y no como un instrumento de diagnóstico clínico.

La evaluación final se realizó sobre un conjunto independiente que no participó en la selección del modelo ni del umbral.


In [19]:

from pathlib import Path
import joblib
import json
from sklearn.base import clone

# Entrenar el modelo definitivo con los 50.000 registros
modelo_final = clone(modelo_rf)
modelo_final.fit(X, y)

# Crear carpeta de modelos
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Guardar pipeline completo (preprocesamiento + Random Forest)
joblib.dump(
    modelo_final,
    MODELS_DIR / "smoking_model.joblib"
)

# Guardar umbral y métricas de evaluación
metadata = {
    "modelo": "RandomForestClassifier",
    "umbral": umbral_final,
    "f1_test": round(float(f1_score(y_test, pred_test)), 4),
    "precision_test": round(float(precision_score(y_test, pred_test)), 4),
    "recall_test": round(float(recall_score(y_test, pred_test)), 4),
    "registros_entrenamiento_final": len(X),
    "variables": X.columns.tolist()
}

with open(MODELS_DIR / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

print("Modelo definitivo guardado correctamente.")
print(f"Registros utilizados: {len(X):,}")
print(f"Variables predictoras: {X.shape[1]}")
print(f"Umbral definitivo: {umbral_final:.2f}")
print("Archivos: smoking_model.joblib y model_metadata.json")


Modelo definitivo guardado correctamente.
Registros utilizados: 50,000
Variables predictoras: 25
Umbral definitivo: 0.41
Archivos: smoking_model.joblib y model_metadata.json
